[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/17_dropout.ipynb)

# 🟢 Easy: Implement Dropout

Реализуйте **Dropout** с нуля. Во время обучения он случайно выключает отдельные элементы активаций, уменьшая зависимость модели от конкретных признаков. Во время инференса Dropout ничего не меняет.

### Inverted Dropout
Пусть элемент сохраняется с вероятностью `1-p`. Если просто занулять элементы, среднее значение активаций уменьшится. Поэтому сохранённые элементы масштабируют:

$$y = \frac{m \odot x}{1-p}, \qquad m_i \sim \operatorname{Bernoulli}(1-p)$$

Тогда `E[y] = x`, и в режиме `eval` не требуется дополнительного масштабирования.

### Режимы модуля
- `module.train()` устанавливает `self.training = True`: нужно создать новую случайную маску на каждом forward;
- `module.eval()` устанавливает `self.training = False`: нужно вернуть вход без изменений;
- для этого упражнения разумный диапазон `p` — `0 <= p < 1`.

### План реализации
1. Сохраните и проверьте вероятность `p` в конструкторе.
2. В `eval` верните вход как identity.
3. В `train` создайте случайную маску той же формы и на том же device.
4. Примените маску и inverted scaling, не отсоединяя результат от autograd.

### Контракт
```python
class MyDropout(nn.Module):
    def __init__(self, p: float = 0.5): ...
    def forward(self, x: Tensor) -> Tensor: ...
```

### Ограничения
- в `train`: зануляйте элементы с вероятностью `p`, остальные масштабируйте на `1/(1-p)`;
- в `eval`: identity;
- не используйте `nn.Dropout` или `F.dropout`.

### Быстрые самопроверки
- при `p=0` train-выход равен входу;
- при `p=0.5` ненулевые элементы единичного входа равны `2`;
- на большом тензоре доля нулей примерно равна `p`;
- при фиксированном `torch.manual_seed` результат воспроизводим;
- backward заполняет градиент входа.

### Типичные ошибки
- применение Dropout в `eval`;
- отсутствие коэффициента `1/(1-p)`;
- одна маска на все вызовы;
- маска на CPU для входа на другом device.

### Официальные материалы и примеры
- [`torch.nn.Dropout`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html) — поведение train/eval, формула и пример;
- [`torch.nn.Module`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html) — базовый класс и флаг `training`.

### Вопросы для самопроверки
1. Почему маску нужно генерировать заново на каждом forward?
2. Зачем масштабировать сохранённые значения?
3. Чем `eval()` отличается от отключения autograd?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

class MyDropout(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        pass

    def forward(self, x):
        pass

In [ ]:
# 🧪 Debug
d = MyDropout(p=0.5)
d.train()
x = torch.ones(10)
print('Train:', d(x))
d.eval()
print('Eval: ', d(x))

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('dropout')